In [ ]:
import pandas as pd 
import plotly.express as px
from plotly.subplots import make_subplots
import plotly.graph_objects as go

In [ ]:
df = pd.read_csv('./data.csv')

# df.info()
# Column name renaming, Change date to actual pandas date, change price to float, and drop or fill na values 
# df.tail(10) # Drop the unnamed column, it is trying to be the index, rename make to manufacture, year

# df['Build'].unique() the build column only has two categories i.e nan and SUV not sedan and wagons etc

# df['Fuel'].unique() has values petrol, nan, diesel, hybrid and electric 

# df['Condition'].unique() Nigerian Used', nan, 'Foreign Used', 'Brand New

# df['Transmission'].unique() Automatic', 'Manual', nan, 'AMT', 'CVT

# df.info()


Drop the Unnamed and Build column 

In [ ]:
df.drop(['Unnamed: 0', 'Build'], axis=1, inplace=True)

Rename columns

In [ ]:
df.rename(columns={
    "Year of manufacture": "Year",
    "Make": "Manufacturer"
}, inplace=True)


df.rename(columns={
  "Engine Size":"engine_size"
}, inplace=True)
# df.head()

Handle NAN

In [ ]:
# [df['Transmission'].mode(),
# df['Condition'].mode(),
# df['Fuel'].mode(),
# df['Mileage'].median()]

# df['Transmission'].mode().iloc[0]

def find_stats(df):
    transmission_mode = df['Transmission'].mode().iloc[0]
    condition_mode = df['Condition'].mode().iloc[0]
    fuel_mode = df['Fuel'].mode().iloc[0]
    mileage_median = df['Mileage'].median()

    return transmission_mode, condition_mode, fuel_mode, mileage_median


transmission, condition, fuel, mileage = find_stats(df)

df['Transmission'] = df['Transmission'].fillna(transmission)
df['Condition'] = df['Condition'].fillna(condition)
df['Fuel'] = df['Fuel'].fillna(fuel)
df['Mileage'] = df['Mileage'].fillna(mileage)
df['engine_size'] = df['engine_size'].fillna(df['engine_size'].median())
df = df.dropna(subset=['Year'])

df['Price'] = df['Price'].str.replace(',','').astype('float')
df['Year'] = df['Year'].astype('int64')

# df.info()

get the age of the car 

In [ ]:
df['car_age'] = df['Year'].max() - df['Year']

# df.head(20)

In [ ]:
df[['Price', 'Mileage', 'engine_size', 'car_age']].corr()

Check Different Conditions and Price Relationship and then visualize them

In [ ]:

# Get the Series of The count of the Manufacturers
manu_count = df['Manufacturer'].value_counts()

# filter through the series to get the manufacturers that are less than 20 and get there index
valid_manufacturers = manu_count[manu_count >= 20].index

# check against the index for filtering and return a data frame of filtered data
filtered_df = df[df['Manufacturer'].isin(valid_manufacturers)]

# condition_data =df.groupby('Condition')['Price'].mean()
# manufacturer_data  =filtered_df.groupby('Manufacturer')['Price'].mean().sort_values(ascending=False)
# fuel_data = df.groupby('Fuel')['Price'].mean()
# transmission_data = df.groupby('Transmission')['Price'].mean()

In [ ]:
# Reset Index adds indexes and turns it into a table
condition_avg = filtered_df.groupby('Condition')['Price'].mean().reset_index()
fuel_avg = filtered_df.groupby('Fuel')['Price'].mean().reset_index()
transmission_avg = filtered_df.groupby('Transmission')['Price'].mean().reset_index()
manufacturers_avg = filtered_df.groupby('Manufacturer')['Price'].mean().sort_values(ascending=False).reset_index()

fig = make_subplots(
    rows=3, cols=2,
    subplot_titles=['Price by Condition', 'Price by Fuel',
                    'Price by Transmission', 'Price by Manufacturer',
                    'Price vs Car Age'],
    specs=[
        [{"type": "bar"}, {"type": "bar"}],
        [{"type": "bar"}, {"type": "bar"}],
        [{"type": "scatter", "colspan": 2}, None]
    ]
)

# Add the Condition Bar Chart
fig.add_trace(
    go.Bar(x=condition_avg['Condition'],
           y=condition_avg['Price'], name='Condition'),
    row=1, col=1
)

# Add Fuel 
fig.add_trace(
  go.Bar(x=fuel_avg['Fuel'], y=fuel_avg['Price'], name = 'Fuel'),
  row=1, col=2
)
# Add Transmission
fig.add_trace(
  go.Bar(x=transmission_avg['Transmission'], y=transmission_avg['Price'], name = 'Transmission'),
  row=2, col=1
)
# Add Manufacturer
fig.add_trace(
  go.Bar(x=manufacturers_avg['Price'], y=manufacturers_avg['Manufacturer'],orientation= 'h', name = 'Manufacturer'),
  row=2, col=2
)

# SHow Relationship between the car age and the price
fig.add_trace(
    go.Scatter(x=df['car_age'], y=df['Price'],
               mode='markers', name='Car Age vs Price'),
    row=3, col=1
)

# fix the ylabel tickers 
fig.update_yaxes(tickprefix='₦', row=1, col=1)
fig.update_yaxes(tickprefix='₦', row=1, col=2)
fig.update_yaxes(tickprefix='₦', row=2, col=1)
# ticker for horizontal bar chart
fig.update_xaxes(tickprefix='₦', row=2, col=2)
# ticker for scatter plot
fig.update_yaxes(tickprefix='₦', row=3, col=1)

# fix the layout issues
fig.update_layout(
    height=1200,
    width=1000,
    title_text='Nigerian Car Market Analysis',
    title_x=0.5,
    showlegend=False,
    template='plotly_dark',
    margin=dict(l=50, r=50, t=80, b=50),
    font= dict(family='Monaco', size=12)
)

fig.write_image('car-analysis.png')
fig.write_html('car-analysis.html')

# fig.show()

Scikit Learn Section for ML and Supervised learning, trying to predict.

In [ ]:
ndf = df.copy()

ndf = ndf.drop(columns=['Mileage','Year'])

ndf = pd.get_dummies(ndf, columns=['Manufacturer','Fuel',"Condition","Transmission"], drop_first=True)



# ndf.head(3)


the steps are 
1. get X and y
2. Split x and y into training set and test set using the train_test_split (usually take 20% for testing)
3. Instantiate a model Object

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error


X = ndf.drop(columns=['Price'])
y = ndf['Price']


X_train,X_test,y_train, y_test = train_test_split(X,y, test_size=0.2, random_state=42)

model = LinearRegression() 



4. fit the x and y training data into it 
5. predict the outcome of y based on x test data 
6. Get the mean absolute error (how far of the model is )

'''
  Actual:    ₦3,200,000
Predicted: ₦2,800,000
Error:       ₦400,000

MAE is 400,000 the lower the better 
'''
7. Get the R2 Score using x and y test data: R**2 value explain how much better a model is at explaining the outcome of y 1.0 is good below 0.5 is bad , ideally 0.7 is ok

These are the steps in using the scikit-learn api

In [ ]:
from sklearn.ensemble import RandomForestRegressor


model = RandomForestRegressor(n_estimators=500, random_state=42)

model.fit(X_train, y_train)

y_pred = model.predict(X_test)

mae = mean_absolute_error(y_test, y_pred)

print(f"MAE: ₦{mae:,.0f}")
print(f"R² Score: {model.score(X_test, y_test):.3f}")


In [ ]:
from xgboost import XGBRegressor

# model = XGBRegressor()
model = XGBRegressor(n_estimators=500, learning_rate=0.05)


model.fit(X_train, y_train)

y_pred = model.predict(X_test)

mae = mean_absolute_error(y_test, y_pred)

print(f"MAE: ₦{mae:,.0f}")
print(f"R² Score: {model.score(X_test, y_test):.3f}")

In [ ]:
importances = pd.Series(model.feature_importances_, index=X.columns)
# importances.sort_values(ascending=True)
important_features = importances[importances > 0.005].index.tolist()

X_train_selected = X_train[important_features]
X_test_selected = X_test[important_features]

model_selected = XGBRegressor(
    n_estimators=500, learning_rate=0.05, random_state=42)
model_selected.fit(X_train_selected, y_train)

y_pred_selected = model_selected.predict(X_test_selected)
mae_selected = mean_absolute_error(y_test, y_pred_selected)

# print(f"MAE: ₦{mae_selected:,.0f}")
# print(f"R² Score: {model_selected.score(X_test_selected, y_test):.3f}")

print(f"Train R²: {model_selected.score(X_train_selected, y_train):.3f}")
print(f"Test R²: {model_selected.score(X_test_selected, y_test):.3f}")

In [22]:
import joblib


joblib.dump(model_selected,'car_predictor.pkl')

['car_predictor.pkl']